# MODFLOW 6: simulación de flujo + un pozo


## Resumen.
Se realiza la misma simulación de flujo realizada en la notebook [06_GWF_json_w.ipynb](06_GWF_json_w.ipynb) pero ahora se leen datos de la simulación desde archivos en formato JSON.

<p xmlns:cc="http://creativecommons.org/ns#" xmlns:dct="http://purl.org/dc/terms/"><a property="dct:title" rel="cc:attributionURL" href="https://github.com/luiggix/mf6_tutorial/">MODFLOW 6: tutorial</a> (00_mf6/07_GWF_json_r.ipynb) by <b>Luis M. de la Cruz Salas (2025)</b> is licensed under <a href="http://creativecommons.org/licenses/by-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">Attribution-ShareAlike 4.0 International<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a>.</p> 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import flopy
import os
import xmf6

Los datos de la inicialización están en el archivo `sim_flow.json`. Usaremos la función `json.load()` para leer esta información en un diccionario general de nombre `sim_flow`:

In [ ]:
with open("sim_flow.json", 'r') as f:
    sim_flow = json.load(f)

# Imprimimos el contenido del diccionario generado
for k, v in sim_flow.items():
    xmf6.nice_print(v, "Datos : " + k)

De igual manera, leemos la información del modelo de flujo del archivo `gwf_d.json`:

In [ ]:
with open("gwf_d.json", 'r') as f:
    gwf_d = json.load(f)
    
for k, v in gwf_d.items():
    xmf6.nice_print(v, "Datos : " + k)

Observamos que hace falta la información de las condiciones de frontera, las propiedades de flujo y los pozos. Esta información se puede agregar como sigue:

In [ ]:
# Parámetros de la simulación (flopy.mf6.MFSimulation)
# Cambiamos el valor del directorio de trabajo en el diccionario 'init' de 'sim_flow'
sim_flow['init']['sim_ws'] = "sandbox7" 

nrow = gwf_d["dis"]["nrow"]
ncol = gwf_d["dis"]["ncol"]
nlay = gwf_d["dis"]["nlay"]

# Parámetros para las condiciones de frontera (flopy.mf6.ModflowGwfchd)
chd_data = []
for row in range(nrow):
    chd_data.append([(0, row, 0), 10.0])       # Condición en la pared izquierda
    chd_data.append([(0, row, ncol - 1), 5.0]) # Condición en la pared derecha
    
gwf_d["chd"]['stress_period_data'] = chd_data # nueva entrada en el diccionario `chd` de `gwf_d`
    
# Parámetros para las propiedades de flujo (flopy.mf6.ModflowGwfnpf)
k_data = np.random.rand(nlay, nrow, ncol)*1.0 
k_data[:, nrow//6 : nrow * 5 // 6, ncol // 3 : ncol * 2 // 3] = 0.1

gwf_d["npf"]['k'] = k_data # nueva entrada en el diccionario `npf` de `gwf_d`

# Parámetros para las propiedades de los pozos (flopy.mf6.ModflowGwfwel)
well_data = [((0, nrow // 2, ncol * 2 // 3), -1.0)]

gwf_d["well"]['stress_period_data'] = well_data # nueva entrada en el diccionario `well` de `gwf_d`

Con toda la información completa, podemos inicializar la simulación, agregar los paquetes, escribir los archivos de entrada y ejecutar la simulación:

In [ ]:
# --- Inicialización de la simulación ---
o_sim = xmf6.common.init_sim(silent = True, **sim_flow)
o_gwf, packages = xmf6.gwf.set_packages(o_sim, silent = True, **gwf_d)

# --- Escritura de archivos ---
o_sim.write_simulation(silent = True)

# --- Ejecución de la simulación ---
o_sim.run_simulation(silent = True)

In [ ]:
# --- Recuperamos los resultados de la simulación ---
head = xmf6.gwf.get_head(o_gwf)
qx, qy, qz, n_q = xmf6.gwf.get_specific_discharge(o_gwf, text="DATA-SPDIS")

# --- Parámetros para las gráficas ---
grid = o_gwf.modelgrid
x, y, z = grid.xyzcellcenters
xticks = np.linspace(grid.extent[0], grid.extent[1], 7)
yticks = np.linspace(grid.extent[2], grid.extent[3], 5)
xlabels = [f'{x:1.1f}' for x in xticks]
ylabels = [f'{y:1.1f}' for y in yticks]
kvmin = 1.0 #np.nanmin(k_data)
kvmax = 0.0 #np.nanmax(k_data)
hvmin = np.nanmin(head)
hvmax = np.nanmax(head)
qvmin = 0.00 #np.nanmin(n_q)
qvmax = 0.35 #np.nanmax(n_q)

# --- Definición de la figura ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize =(10,10))

# --- Gráfica 1. ---
kview = flopy.plot.PlotMapView(model = o_gwf, ax = ax1)
kview.plot_grid(linewidths = 0.5, alpha = 0.5)
k_ac = kview.plot_array(k_data, cmap = "gist_heat", vmin = kvmin, vmax = kvmax)
k_cb = plt.colorbar(k_ac, ax = ax1, label = "$k$", 
                    ticks = [0.0, 0.25, 0.50, 0.75, 1.0],
                    cax = xmf6.vis.cax(ax1, k_ac))
k_cb.ax.tick_params(labelsize=8)
ax1.set_title("Permeabilidad hidráulica $k$", fontsize=10)
ax1.set_ylabel("$y$ (m)", fontsize = 8)
ax1.set_xticks(ticks = xticks, labels = xlabels, fontsize = 8)
ax1.set_yticks(ticks = yticks, labels = ylabels, fontsize = 8)
ax1.set_aspect('equal')

# --- Gráfica 2. ---
hview = flopy.plot.PlotMapView(model = o_gwf, ax = ax2)
h_ac = hview.plot_array(head, cmap = "YlGnBu", vmin = hvmin, vmax = hvmax, alpha = 0.75)
hview.contour_array(head, levels = 30, cmap = "bone", linewidths = 1.0)
ax2.quiver(x, y, qx[0], qy[0], scale = 3, 
           color = 'k', linewidth = 0.95, pivot = 'middle')
h_cb = plt.colorbar(h_ac, ax = ax2, label = "$h$ (m)", 
                    cax = xmf6.vis.cax(ax2, h_ac))
h_cb.ax.tick_params(labelsize=8)
ax2.set_title("Carga hidráulica $h$", fontsize=10)
ax2.set_ylabel("$y$ (m)", fontsize = 8)
ax2.set_xticks(ticks = xticks, labels = xlabels, fontsize = 8)
ax2.set_yticks(ticks = yticks, labels = ylabels, fontsize = 8)
ax2.set_aspect('equal')

# --- Gráfica 3. ---
fview = flopy.plot.PlotMapView(model = o_gwf, ax = ax3)
q_ac = fview.plot_array(n_q, cmap = "winter", vmin = qvmin, vmax = qvmax, alpha = 0.25)
fview.contour_array(head, levels = 20, cmap = 'bone', linewidths = 0.75, )
ax3.streamplot(x, y[::-1][:], qx[0], qy[0][::-1], 
               density = [2, 1.5], linewidth = 0.75, broken_streamlines = True, 
               color = n_q, cmap = "winter", 
               arrowstyle = "->", arrowsize = 0.75,  )
q_cb = plt.colorbar(q_ac, ax=ax3, label="$q$", 
                    ticks = np.linspace(0.0, 0.35, 7),
                    format = "{x:3.2f}",
                    cax = xmf6.vis.cax(ax3, q_ac))
q_cb.ax.tick_params(labelsize=8)
ax3.set_title("Descarga específica $q$", fontsize=10)
ax3.set_xlabel("$x$ (m)", fontsize = 8)
ax3.set_ylabel("$y$ (m)", fontsize = 8)
ax3.set_xticks(ticks = xticks, labels = xlabels, fontsize = 8)
ax3.set_yticks(ticks = yticks, labels = ylabels, fontsize = 8)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()